# Structured Output with TypedDict

## What is TypedDict?

`TypedDict` is a built-in Python type (from the `typing` module) for defining dictionaries with known keys and typed values. It's simpler than Pydantic — no validation, just a type hint.

```python
from typing import TypedDict

class User(TypedDict):
    name: str
    age: int
    email: str

# TypedDict is just a dict with known keys:
user: User = {"name": "Alice", "age": 30, "email": "alice@example.com"}
print(user["name"])  # "Alice"
```

## TypedDict vs. Pydantic BaseModel

| Feature | TypedDict | Pydantic BaseModel |
|---------|-----------|-------------------|
| **Output type** | `dict` — access with `["key"]` | Object — access with `.key` |
| **Validation** | None (wrong types accepted silently) | Strict — raises error on wrong type |
| **Performance** | Faster (no validation overhead) | Slightly slower |
| **Best for** | Quick schemas, LangChain structured output | Production code, APIs, data pipelines |
| **Extra features** | None | `Field()`, validators, `.model_dump_json()`, etc. |

## Using TypedDict with `with_structured_output()`

Just like Pydantic, TypedDict can be passed to `model.with_structured_output()`:

```python
from typing import TypedDict, Annotated, Literal

class Review(TypedDict):
    sentiment: Annotated[Literal["positive", "negative"], "The sentiment"]
    summary: Annotated[str, "A brief summary"]

model = ChatOllama(model="qwen2.5:latest")
structured_model = model.with_structured_output(Review)
result = structured_model.invoke("This product is terrible!")
# result is a plain dict:
print(result["sentiment"])  # "negative"
```

## What you'll learn in this notebook

- How to define a `TypedDict` schema
- How TypedDict differs from Pydantic in practice
- How to use `Annotated` to add field descriptions for the LLM
- A live example connecting TypedDict to a real model

## Prerequisites

- Ollama running with a model pulled (the LLM demo at the end uses `qwen2.5:latest`)
- Virtual environment activated

In [1]:
from typing import TypedDict

In [2]:
class User(TypedDict):
    name: str
    age: int
    email: str
    

In [3]:
user = User(name="Alice", age=30, email="rj.rahul.jauhari@gmail.com")

In [4]:
user

{'name': 'Alice', 'age': 30, 'email': 'rj.rahul.jauhari@gmail.com'}

In [ ]:
# --- Section: TypedDict with a Real LLM ---
# Now let's connect TypedDict to an actual model using with_structured_output().

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated, Literal

# Define a schema using TypedDict
class MovieReview(TypedDict):
    title: Annotated[str, "The title of the movie being reviewed"]
    sentiment: Annotated[Literal["positive", "negative", "neutral"], "Overall sentiment of the review"]
    one_line_summary: Annotated[str, "A single sentence summarising the review"]

# Attach the schema to the model — it will now always return data in this shape
model = ChatOllama(model="qwen2.5:latest")  # change to any model you have
structured_model = model.with_structured_output(MovieReview)

review_text = "Inception is an absolute masterpiece. Nolan's direction is flawless and the concept is mind-blowing."

result = structured_model.invoke(review_text)

# result is a plain Python dict
print(f"Title:    {result['title']}")
print(f"Sentiment: {result['sentiment']}")
print(f"Summary:  {result['one_line_summary']}")